# Rubin DP2 six-band deconvolution placeholder → 4 × 3 predicted IRG grids

Batch version of the **section 6 / 6b workflow in `lantern_individual_target_html_viewer_v21(2).ipynb`**. Previous name `lantern_dp2_predicted_irg_grid_cached.ipynb`.

1. Accept a list of `target_id` values **or** `(RA, Dec)` coordinates.
2. For each target, retrieve actual Rubin DP2 **deep-coadd Exposure images and local PSFs** using v21's SIA + SODA approach.
3. Construct `(6, 32, 32)` `observed_object_images` and `observed_psf_images` in **`u, g, r, i, z, y`** order, with v21's nearest-available-band fallback applied **only after actual image and PSF retrieval**.
4. Call a **placeholder only** `deconvolve_multiband(...)` returning `(6,32,32)` `predicted_object_images` and `predicted_psf_images`. The placeholder is **identity/copy**, not deconvolution; replace its body with your model.
5. Build the **predicted IRG color images** from `predicted_object_images`: **i → R, r → G, g → B**; display 4 columns × 3 rows per page.

The predicted IRG display uses **fixed, linear scaling per RGB band** (`i → R`, `r → G`, `g → B`), with independent editable `vmin/vmax` values for each band. Each band keeps the same fixed limits for every candidate, including across separate batches. No percentile scaling or asinh stretch is used.

**Run on Rubin Science Platform (RSP)** with DP2 access. This notebook does not fetch alerts, thumbnails, or Legacy Survey images. RSP retrieval cannot be run offline. All six observed and predicted stacks can optionally be saved per target as `.npz` files.

**Caching:** Set `LOAD_NPZ_IF_EXISTS=True` to reuse saved six-band observed and predicted arrays. Cached targets skip Rubin downloads **and** the deconvolution placeholder. `SAVE_NPZ` independently controls whether newly processed targets are saved. Cache entries are verified against the target coordinates to avoid mixing targets.


In [ ]:
# ============================================================
# USER SETTINGS
# ============================================================
from pathlib import Path
import numpy as np

# Choose EXACTLY ONE input style.
TARGET_IDS = [
    # "target_1",
    # "target_2",
    "target_0002330",
    "target_0005448",
    "target_0012511",
    "target_0024385",
    "target_0010393",
    "target_0000641",
    "target_0012310",
    "target_0030455",
    #"target_0020630",
    "target_0009557",
]
TARGET_IDS0 = [
    # "target_1",
    # "target_2",
    "target_0002330",
    "target_0002938",
    "target_0005448",
    "target_0009843",
    "target_0024750",
    "target_0029791",
    "target_0033124",
    "target_0034363",
    "target_0008756",
    "target_0025823",
    "target_0012511",
    "target_0024385",
    "target_0009563",
    "target_0010393",
]

TARGET_IDS1 = [
    "target_0039816",
    "target_0011102",
    "target_0000641",
    "target_0010498",
    "target_0012310",
    "target_0010654",
    "target_0024105",
    "target_0030455",
    "target_0015373",
    "target_0020630",
    "target_0005487",
    "target_0007636",
    "target_0011540",
    "target_0029368",
    "target_0039871",
    "target_0005743",
    "target_0030276",
    "target_0008107",
    "target_0031536",
    "target_0008859",
    "target_0009557",
]

# (RA_deg, Dec_deg) or (label, RA_deg, Dec_deg).
COORDINATES = [
    # (53.123456, -27.123456),
    # ("candidate_B", 53.234567, -27.234567),
]

# Same candidate table as v21; only used for TARGET_IDS.
CANDIDATES_FILE = Path(
    "candidate_analysis_hybrid_multi_loci_v8_no_gaia_color_cut/"
    "second_pass_candidates_with_flags.parquet"
)

OUTPUT_DIR = Path("deconvolution_placeholder_irg_grid")
LOAD_NPZ_IF_EXISTS = True  # Reuse saved six-band tensors; skip slow Section 4 work
SAVE_NPZ = True  # Save newly processed targets; independent of loading
SAVE_PNG = True
SAVE_HTML = True

# V21 exposure + PSF retrieval; cutout radius 0.001 deg = 3.6 arcsec.
COADD_CUTOUT_RADIUS_DEG = 0.001
DECONV_CROP_SIZE = 32

# Fixed linear display limits, in RGB order: i (red), r (green), g (blue).
# Adjust each band independently; these values are identical for all targets/runs.
# Example: increase the i-band vmax relative to the others to reduce apparent redness.
DISPLAY_VMIN = [-100.0, -100.0, -100.0]
DISPLAY_VMAX = np.array([3000.0, 2500.0, 1500.0])*0.7

# No additional color manipulation for strictly linear R/G/B display.
RGB_SATURATION = 1.0

N_COLUMNS = 3 #4
N_ROWS = 3
PER_PAGE = N_COLUMNS * N_ROWS

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ============================================================
# IMPORTS (run in Rubin RSP environment)
# ============================================================
import base64
import html
import io
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageEnhance

from astropy import units as u
from astropy.coordinates import SkyCoord
from lsst.images.serialization import read_archive
from lsst.rsp import RSPDiscovery
from lsst.rsp.utils import get_pyvo_auth
from pyvo.dal.adhoc import SodaQuery

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

In [ ]:
sys.path.append('/home/sfu/SL_time_delay_2026a/loci/SharPy')
from sharpy.enhancement import Enhancement

my_enh = Enhancement(device='GPU', trained_model_file=None)

## 1. Resolve target IDs or RA/Dec

As in the earlier grid, coordinates need no candidate file. Missing or ambiguous IDs are recorded as failures so the grid can still display the requested ordering.

In [ ]:
def prepare_targets():
    if bool(TARGET_IDS) == bool(COORDINATES):
        raise ValueError("Fill exactly one of TARGET_IDS or COORDINATES.")

    rows = []
    if TARGET_IDS:
        if not CANDIDATES_FILE.exists():
            raise FileNotFoundError(f"Candidate table missing: {CANDIDATES_FILE}")
        df = pd.read_parquet(CANDIDATES_FILE, columns=["target_id", "ra", "dec"]).copy()
        df["_id"] = df["target_id"].astype(str)
        lookup = {key: group for key, group in df.groupby("_id", sort=False)}
        for tid in TARGET_IDS:
            tid = str(tid)
            matches = lookup.get(tid)
            if matches is None or len(matches) != 1:
                issue = "ID not found" if matches is None else f"Ambiguous ID: {len(matches)} rows"
                rows.append(dict(label=tid, target_id=tid, ra=np.nan,
                                 dec=np.nan, input_error=issue))
            else:
                m = matches.iloc[0]
                rows.append(dict(label=tid, target_id=tid, ra=m["ra"],
                                 dec=m["dec"], input_error=""))
    else:
        for k, entry in enumerate(COORDINATES, start=1):
            if len(entry) == 2:
                ra, dec = entry
                label = f"coord_{k:03d}"
            elif len(entry) == 3:
                label, ra, dec = entry
            else:
                raise ValueError(f"Invalid COORDINATES entry: {entry}")
            rows.append(dict(label=str(label), target_id="", ra=ra,
                             dec=dec, input_error=""))

    for row in rows:
        try:
            ra, dec = float(row["ra"]), float(row["dec"])
            if not (np.isfinite(ra) and np.isfinite(dec) and
                    0 <= ra < 360 and -90 <= dec <= 90):
                raise ValueError("RA/Dec invalid or outside allowed range")
            row["ra"], row["dec"] = ra, dec
        except (ValueError, TypeError) as exc:
            row.update(ra=np.nan, dec=np.nan,
                       input_error=row["input_error"] or str(exc))
    return rows


targets = prepare_targets()
print(f"Targets: {len(targets)}, pages: {(len(targets) + PER_PAGE - 1) // PER_PAGE}")
display(pd.DataFrame(targets))

## 2. Rubin DP2 SIA and SODA Exposure + PSF retrieval

This follows v21: one SIA search per position for `lsst.deep_coadd`, then SODA `cutout-sync-exposure`, deserialize the full Exposure with `read_archive`, and evaluate its local PSF model at the target position. A band is **usable only if its image and PSF both succeed**. Any missing/failed requested band is replaced with the nearest usable band in `ugrizy`, with ties resolved toward the middle (v21's rule).

In [ ]:
BAND_ORDER = list("ugrizy")
BAND_INDEX = {band: i for i, band in enumerate(BAND_ORDER)}
BAND_CENTER_INDEX = (len(BAND_ORDER) - 1) / 2.0


def choose_source_band(requested_band, usable_bands):
    """v21: exact band, otherwise nearest ugrizy band, tie to middle."""
    if requested_band in usable_bands:
        return requested_band
    if not usable_bands:
        raise ValueError("No usable bands")
    req_i = BAND_INDEX[requested_band]
    return sorted(
        usable_bands,
        key=lambda b: (abs(BAND_INDEX[b] - req_i),
                       abs(BAND_INDEX[b] - BAND_CENTER_INDEX),
                       BAND_INDEX[b]),
    )[0]


def center_crop_or_pad(array, size=32, fill_value=0.0):
    """v21 centered crop/pad, applied separately to image and PSF."""
    array = np.asarray(array, dtype=float)
    if array.ndim != 2:
        raise ValueError("Input image/PSF must be 2D")
    ny, nx = array.shape
    out = np.full((size, size), fill_value, dtype=float)
    yi, xi = max(0, (ny-size)//2), max(0, (nx-size)//2)
    crop = array[yi:min(ny, yi+size), xi:min(nx, xi+size)]
    cy, cx = crop.shape
    yo, xo = (size-cy)//2, (size-cx)//2
    out[yo:yo+cy, xo:xo+cx] = crop
    return out


def _get_exposure_image_array(exposure):
    if hasattr(exposure, "image"):
        return np.asarray(exposure.image.array, dtype=float)
    if hasattr(exposure, "array"):
        return np.asarray(exposure.array, dtype=float)
    raise AttributeError("Exposure has no image pixel array")


def _target_pixel(exposure, ra, dec):
    try:
        pos = exposure.sky_projection.sky_to_pixel(
            SkyCoord(ra=ra*u.deg, dec=dec*u.deg, frame="icrs")
        )
        return float(pos.x), float(pos.y)
    except Exception:
        bbox = exposure.bbox
        return (0.5 * (bbox.x.min + bbox.x.max),
                0.5 * (bbox.y.min + bbox.y.max))


def _get_local_psf_array(exposure, ra, dec):
    x, y = _target_pixel(exposure, ra, dec)
    psf = np.asarray(exposure.psf.compute_kernel_image(x=x, y=y).array, dtype=float)
    total = np.nansum(psf)
    if not np.isfinite(total) or total <= 0:
        raise ValueError("PSF sum is non-positive/non-finite")
    return psf / total


def _fetch_single_exposure(coadd_search, coadd_table, band, ra, dec, discovery):
    """Try matching SIA rows, ordered with v21's nearest footprint center first."""
    band_values = np.asarray(coadd_table["lsst_band"]).astype(str)
    indices = np.where(band_values == band)[0]
    if not len(indices):
        raise ValueError(f"No SIA rows for {band}")

    def center_distance(index):
        try:
            s_ra = float(coadd_table["s_ra"][index])
            s_dec = float(coadd_table["s_dec"][index])
            return np.hypot(s_ra - ra, s_dec - dec)
        except Exception:
            return np.inf

    indices = sorted((int(i) for i in indices), key=center_distance)
    failures = []
    for idx in indices:
        try:
            dl_result = discovery.get_datalink_results(coadd_search[idx])
            sq = SodaQuery.from_resource(
                dl_result,
                dl_result.get_adhocservice_by_id("cutout-sync-exposure"),
                session=get_pyvo_auth(),
            )
            sq.circle = (ra*u.deg, dec*u.deg, COADD_CUTOUT_RADIUS_DEG*u.deg)
            data = sq.execute_stream().read()
            sq.raise_if_error()
            exposure = read_archive(io.BytesIO(data))
            image = _get_exposure_image_array(exposure)
            psf = _get_local_psf_array(exposure, ra, dec)
            if image.ndim != 2 or psf.ndim != 2:
                raise ValueError("Image or PSF is not 2D")
            if not np.any(np.isfinite(image)) or not np.any(np.isfinite(psf)):
                raise ValueError("Image or PSF has no finite pixels")
            return image, psf
        except Exception as exc:
            failures.append(f"SIA row {idx}: {exc}")
    raise RuntimeError(f"All Exposure/PSF retrievals failed for {band}: " +
                       " | ".join(failures))


def fetch_six_band_observed(ra, dec, sia_client, discovery):
    coadd_search = sia_client.search(
        pos=(ra, dec, COADD_CUTOUT_RADIUS_DEG),
        calib_level=3,
        dpsubtype="lsst.deep_coadd",
    )
    coadd_table = coadd_search.to_table()
    if len(coadd_table) == 0:
        raise RuntimeError("No deep_coadd SIA results")
    available = [b for b in BAND_ORDER if b in
                 set(np.asarray(coadd_table["lsst_band"]).astype(str))]
    if not available:
        raise RuntimeError("No recognizable ugrizy deep-coadd bands")

    retrieved = {}
    band_errors = {}
    for band in available:
        try:
            retrieved[band] = _fetch_single_exposure(
                coadd_search, coadd_table, band, ra, dec, discovery
            )
        except Exception as exc:
            band_errors[band] = str(exc)

    usable = [b for b in BAND_ORDER if b in retrieved]
    if not usable:
        raise RuntimeError(f"No band has both image and PSF: {band_errors}")

    # IMPORTANT: fallback is chosen AFTER checking that both arrays were retrieved.
    mapping = {b: choose_source_band(b, usable) for b in BAND_ORDER}
    observed_images, observed_psfs = [], []
    for band in BAND_ORDER:
        image, psf = retrieved[mapping[band]]
        im32 = center_crop_or_pad(image, DECONV_CROP_SIZE)
        psf32 = center_crop_or_pad(psf, DECONV_CROP_SIZE)
        psf_sum = np.nansum(psf32)
        if not np.isfinite(psf_sum) or psf_sum <= 0:
            raise RuntimeError(f"PSF after cropping invalid for {band}")
        observed_images.append(im32)
        observed_psfs.append(psf32 / psf_sum)

    return (np.stack(observed_images, axis=0),
            np.stack(observed_psfs, axis=0), mapping, band_errors)

## 3. Replaceable deconvolution placeholder (same inputs/outputs as v21)

**No Richardson–Lucy or other deconvolution is performed.** Edit only the marked two lines when connecting your own six-band predictor. By default, the **predicted** arrays are *copies* of the observed arrays, so the grid is **not actually deconvolved** until you replace the placeholder.

In [ ]:
def deconvolve_multiband(observed_object_images, observed_psf_images, **kwargs):
    """
    Inputs:
        observed_object_images : (6, 32, 32) in ugrizy order
        observed_psf_images    : (6, 32, 32) in ugrizy order
    Returns:
        predicted_object_images: (6, 32, 32) in ugrizy order
        predicted_psf_images   : (6, 32, 32) in ugrizy order

    PLACEHOLDER ONLY: default output is an identity copy, NOT deconvolution.
    """
    observed_object_images = np.asarray(observed_object_images, dtype=float)
    observed_psf_images = np.asarray(observed_psf_images, dtype=float)
    required = (6, DECONV_CROP_SIZE, DECONV_CROP_SIZE)
    if observed_object_images.shape != required or observed_psf_images.shape != required:
        raise ValueError(f"Expected both observed inputs shaped {required}")

    # ========================================================
    # INSERT YOUR DECONVOLUTION MODEL HERE.
    # Replace ONLY these two identity-copy lines:
    #predicted_object_images = observed_object_images.copy()
    #predicted_psf_images = observed_psf_images.copy()

    predicted_object_images, predicted_psf_images = my_enh.enhance(
        observed_object_images, 
        observed_psf_images,
        fix_psf=False,
    )
    # ========================================================

    return predicted_object_images, predicted_psf_images


def validate_predicted_arrays(images, psfs):
    """Accept (6,32,32), or v21-supported (1,6,32,32)."""
    required = (6, DECONV_CROP_SIZE, DECONV_CROP_SIZE)
    def convert(array, name):
        array = np.asarray(array, dtype=float)
        if array.shape == (1, *required):
            array = array[0]
        if array.shape != required:
            raise ValueError(f"{name} has shape {array.shape}, expected {required}")
        if not np.any(np.isfinite(array)):
            raise ValueError(f"{name} contains no finite pixels")
        return array
    return convert(images, "predicted_object_images"), convert(psfs, "predicted_psf_images")

## 4. Load saved arrays if available; otherwise retrieve / run placeholder / optionally save

When `LOAD_NPZ_IF_EXISTS=True`, previously saved `.npz` files are loaded **before** any Rubin queries or model calls. Both `predicted_object_images` and `predicted_psf_images` are restored, along with the band fallback mapping. The saved coordinates must match the requested candidate to avoid using another target's cache. Cache filenames can be found even if you reorder your target list.

`SAVE_NPZ=False` by itself does **not** load a cache; the new `LOAD_NPZ_IF_EXISTS` switch handles that. For uncached targets, downloading and inference still occur. If you are only changing RGB scaling **and `results` already exists in the active kernel**, you can skip this cell altogether and rerun Sections 5–7.


In [ ]:
def safe_name(value):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(value))


def target_npz_path(item):
    return OUTPUT_DIR / (f"{item['index']:04d}_{safe_name(item['label'])}_deconvolution_arrays.npz")


def save_target_npz(item, arrays, mapping):
    obs_im, obs_psf, pred_im, pred_psf = arrays
    path = target_npz_path(item)
    np.savez_compressed(
        path,
        observed_object_images=obs_im,
        observed_psf_images=obs_psf,
        predicted_object_images=pred_im,
        predicted_psf_images=pred_psf,
        band_order=np.asarray(BAND_ORDER),
        requested_to_source_band=np.asarray([mapping[b] for b in BAND_ORDER]),
        ra_deg=np.asarray(item["ra"]),
        dec_deg=np.asarray(item["dec"]),
    )
    return path


def load_target_npz(item):
    """Load a matching six-band cache; return None if none is valid.

    First try the standard filename. Also look for matching labels under a
    different list index, e.g. when TARGET_IDS have been reordered.
    """
    exact = target_npz_path(item)
    pattern = f"*_{safe_name(item['label'])}_deconvolution_arrays.npz"
    candidates = [exact] + [p for p in sorted(OUTPUT_DIR.glob(pattern)) if p != exact]
    for path in candidates:
        if not path.is_file():
            continue
        try:
            with np.load(path, allow_pickle=False) as saved:
                # No silently reused arrays for a target at another position.
                cached_ra = float(saved["ra_deg"])
                cached_dec = float(saved["dec_deg"])
                if not (np.isclose(cached_ra, item["ra"], rtol=0, atol=1e-7) and
                        np.isclose(cached_dec, item["dec"], rtol=0, atol=1e-7)):
                    continue
                if list(saved["band_order"].astype(str)) != BAND_ORDER:
                    continue
                source_bands = list(saved["requested_to_source_band"].astype(str))
                if len(source_bands) != len(BAND_ORDER) or any(b not in BAND_ORDER for b in source_bands):
                    continue
                observed_object_images = np.asarray(saved["observed_object_images"], dtype=float)
                observed_psf_images = np.asarray(saved["observed_psf_images"], dtype=float)
                predicted_object_images, predicted_psf_images = validate_predicted_arrays(
                    saved["predicted_object_images"], saved["predicted_psf_images"]
                )
                required = (6, DECONV_CROP_SIZE, DECONV_CROP_SIZE)
                if observed_object_images.shape != required or observed_psf_images.shape != required:
                    raise ValueError(f"Saved observed stacks are not {required}")
                mapping = dict(zip(BAND_ORDER, source_bands))
            return (predicted_object_images, predicted_psf_images, mapping, path)
        except (OSError, KeyError, ValueError, TypeError) as exc:
            print(f"  Ignoring invalid cache {path.name}: {exc}")
    return None


# Establish a connection ONLY if an uncached target actually needs downloading.
discovery = None
sia_client = None
results = []
for k, target in enumerate(targets, start=1):
    item = dict(target, index=k, status="error", error="", mapping={},
                predicted_object_images=None, predicted_psf_images=None,
                npz_path="", source="")
    if item["input_error"]:
        item["error"] = item["input_error"]
        results.append(item)
        continue
    print(f"[{k}/{len(targets)}] {item['label']}: RA={item['ra']:.6f} Dec={item['dec']:.6f}")
    try:
        cached = load_target_npz(item) if LOAD_NPZ_IF_EXISTS else None
        if cached is not None:
            pred_im, pred_psf, mapping, path = cached
            item.update(predicted_object_images=pred_im, predicted_psf_images=pred_psf,
                        mapping=mapping, npz_path=str(path), status="ok", source="cache")
            print(f"  Loaded: {path.name} (Rubin retrieval and model skipped)")
        else:
            if discovery is None:
                discovery = RSPDiscovery("dp2")
                sia_client = discovery.get_sia_client()
            obs_im, obs_psf, mapping, errors = fetch_six_band_observed(
                item["ra"], item["dec"], sia_client, discovery
            )
            pred_im, pred_psf = deconvolve_multiband(obs_im, obs_psf)
            pred_im, pred_psf = validate_predicted_arrays(pred_im, pred_psf)
            item.update(predicted_object_images=pred_im, predicted_psf_images=pred_psf,
                        mapping=mapping, fetch_errors=errors, status="ok", source="computed")
            if SAVE_NPZ:
                item["npz_path"] = str(save_target_npz(
                    item, (obs_im, obs_psf, pred_im, pred_psf), mapping
                ))
            print("  Retrieved / processed (no valid cached arrays found)")
        fallback = [f"{b}→{item['mapping'][b]}" for b in BAND_ORDER if item['mapping'][b] != b]
        print("  fallback:", ", ".join(fallback) if fallback else "none")
    except Exception as exc:
        item["error"] = str(exc)
        print("  FAILED:", exc)
    results.append(item)

summary = pd.DataFrame([{
    "index": r["index"], "label": r["label"], "target_id": r["target_id"],
    "ra": r["ra"], "dec": r["dec"], "status": r["status"], "source": r["source"],
    "requested_to_source_band": ", ".join(f"{b}:{r['mapping'][b]}"
                                           for b in BAND_ORDER) if r["mapping"] else "",
    "fallback": ", ".join(f"{b}→{r['mapping'][b]}" for b in BAND_ORDER
                           if r["mapping"] and r["mapping"][b] != b),
    "error": r["error"], "npz_path": r["npz_path"],
} for r in results])
status_path = OUTPUT_DIR / "deconvolution_grid_status.csv"
summary.to_csv(status_path, index=False)
print("Saved status:", status_path)
display(summary)


## 5. Fixed per-band **linear** IRG display using the predicted object tensors

Choose a **single** `DISPLAY_VMAX` (fixed or computed from all predicted IRG pixels) and use it for every candidate **and all three channels**. A smaller shared `vmax` lifts faint details but saturates bright cores; with strictly linear RGB it is not possible to compress bright features independently. No per-frame/channel autostretch and no asinh transform.

In [ ]:
RGB_BANDS = ("i", "r", "g")  # red, green, blue


def select_predicted_irg(predicted_object_images):
    return np.stack([predicted_object_images[BAND_INDEX[b]] for b in RGB_BANDS], axis=-1)


successful = [r for r in results if r["status"] == "ok"]
if not successful:
    raise RuntimeError("No usable candidate. See the status table above.")

# Fixed min/max for each RGB band; numpy broadcasts (3,) across (height, width, 3).
rgb_vmin = np.asarray(DISPLAY_VMIN, dtype=float)
rgb_vmax = np.asarray(DISPLAY_VMAX, dtype=float)
if rgb_vmin.shape != (3,) or rgb_vmax.shape != (3,):
    raise ValueError("DISPLAY_VMIN and DISPLAY_VMAX must each contain 3 values (i, r, g).")
if not np.all(np.isfinite(rgb_vmin)) or not np.all(np.isfinite(rgb_vmax)):
    raise ValueError("All RGB display limits must be finite.")
if np.any(rgb_vmax <= rgb_vmin):
    raise ValueError("DISPLAY_VMAX must exceed DISPLAY_VMIN in each band.")
if RGB_SATURATION != 1.0:
    raise ValueError("Set RGB_SATURATION=1.0 to preserve strictly linear channel scaling.")


def render_predicted_irg(predicted_object_images):
    data = select_predicted_irg(predicted_object_images)
    rgb = np.clip((data - rgb_vmin) / (rgb_vmax - rgb_vmin), 0.0, 1.0)
    return np.nan_to_num(rgb, nan=0.0, posinf=1.0, neginf=0.0)


for item in successful:
    item["rgb"] = render_predicted_irg(item["predicted_object_images"])

print("Fixed linear per-band RGB limits (i, r, g):")
print("  vmin:", rgb_vmin.tolist())
print("  vmax:", rgb_vmax.tolist())
print("Reminder: until you replace the placeholder, predicted arrays = observed arrays.")


## 6. Plot the 4-column × 3-row predicted IRG grids

All panels use the same linear scale. The caption marks any substituted **RGB** bands. Missing targets remain visible as unavailable slots.

In [ ]:
def compact_error(value, n=90):
    value = str(value)
    return value if len(value) <= n else value[:n-3] + "..."


saved_pngs = []
for start in range(0, len(results), PER_PAGE):
    block = results[start:start+PER_PAGE]
    page = start // PER_PAGE + 1
    fig, axes = plt.subplots(N_ROWS, N_COLUMNS, figsize=(9.0, 9.0),#figsize=(12.0, 9.0),
                             squeeze=False, constrained_layout=True)
    for ax in axes.flat:
        ax.set_visible(False)
    for j, item in enumerate(block):
        ax = axes.flat[j]
        ax.set_visible(True)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
        label = compact_error(item["label"], 28)
        #ax.set_title(f"{item['index']}. {label}", fontsize=11, pad=4)
        if item["status"] == "ok":
            ax.imshow(item["rgb"], origin="lower", interpolation="nearest")
            fallback = " ".join(f"{b}→{item['mapping'][b]}" for b in RGB_BANDS
                                if item["mapping"][b] != b)
            if fallback:
                ax.text(0.5, 0.04, fallback, transform=ax.transAxes,
                        ha="center", va="bottom", fontsize=10, color="white",
                        bbox={"facecolor": "black", "alpha": 0.8, "pad": 2})
        else:
            ax.set_facecolor("0.9")
            ax.text(0.5, 0.5, "Unavailable\n" + compact_error(item["error"], 65),
                    transform=ax.transAxes, ha="center", va="center",
                    fontsize=9, wrap=True)
        #if np.isfinite(item["ra"]) and np.isfinite(item["dec"]):
        #    ax.set_xlabel(f"{item['ra']:.5f}, {item['dec']:+.5f} deg", fontsize=9)

    #fig.suptitle(f"DP2 predicted IRG | page {page} | linear fixed per-band scaling"
    #             " | placeholder until model added", fontsize=13)
    if SAVE_PNG:
        path = OUTPUT_DIR / f"predicted_irg_grid_page_{page:02d}.png"
        fig.savefig(path, dpi=170, bbox_inches="tight")
        saved_pngs.append(path)
        print("Saved:", path)
    plt.show()
    plt.close(fig)

## 7. Optional HTML gallery

A self-contained HTML version of the same 4×3 grid, generated **from the predicted images**. The PNGs/HTML are display products; the `.npz` tensors retain full floating-point values.

In [ ]:
def rgb_data_uri(rgb):
    arr = np.uint8(np.rint(255*np.clip(np.flipud(rgb), 0, 1)))
    buf = io.BytesIO()
    Image.fromarray(arr, mode="RGB").save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode("ascii")


if SAVE_HTML:
    pages = []
    for start in range(0, len(results), PER_PAGE):
        cards = []
        for r in results[start:start+PER_PAGE]:
            label = html.escape(str(r["label"]))
            if r["status"] == "ok":
                im = f'<img src="{rgb_data_uri(r["rgb"])}" alt="predicted IRG {label}">'
                fb = ", ".join(f"{b}→{r['mapping'][b]}" for b in RGB_BANDS
                               if r["mapping"][b] != b)
                note = f'<p class="fb">RGB fallback: {html.escape(fb)}</p>' if fb else ""
            else:
                im = '<div class="missing">Cutout unavailable</div>'
                note = f'<p class="error">{html.escape(compact_error(r["error"], 150))}</p>'
            loc = (f"{r['ra']:.6f}, {r['dec']:+.6f} deg"
                   if np.isfinite(r["ra"]) and np.isfinite(r["dec"]) else "Position unknown")
            cards.append(f'<article><b>{r["index"]}. {label}</b>{im}'
                         f'<p>{html.escape(loc)}</p>{note}</article>')
        pages.append(f'<section><h2>Page {start//PER_PAGE+1}</h2><div class="grid">'
                     + "".join(cards) + "</div></section>")
    markup = f"""<!doctype html><html lang="en"><head><meta charset="utf-8">
<title>Rubin DP2 predicted IRG grid</title>
<style>
body{{font-family:Arial,sans-serif;max-width:1300px;margin:20px auto;padding:0 15px}}
h1{{font-size:1.5rem}} .grid{{display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:12px}}
article{{border:1px solid #ddd;padding:9px;min-width:0;overflow-wrap:anywhere}}
article img{{display:block;width:100%;aspect-ratio:1;object-fit:contain;image-rendering:pixelated;background:#000}}
article p{{font-size:.82rem}} .fb{{color:#805900}} .error{{color:#a22}}
.missing{{aspect-ratio:1;display:grid;place-items:center;background:#eee}}
section{{margin:30px 0;break-after:page}}
@media(max-width:720px){{.grid{{grid-template-columns:repeat(2,minmax(0,1fr))}}}}
</style></head><body>
<h1>Rubin DP2 predicted IRG</h1>
<p>Predicted i=red, r=green, g=blue. Fixed linear per-band scaling (i/r/g): vmin={rgb_vmin.tolist()},
vmax={rgb_vmax.tolist()}. Identity placeholder unless model has been inserted.</p>
{''.join(pages)}
</body></html>"""
    path = OUTPUT_DIR / "predicted_irg_grid.html"
    path.write_text(markup, encoding="utf-8")
    print("Saved:", path)

## Notes

- **Not a deconvolution yet:** the default `deconvolve_multiband()` simply copies its inputs; predicted IRG therefore visualizes the cropped/padded deep-coadd images. Replace those two lines with your model and rerun the retrieval/model/rendering cells.
- **Exact six-band interface:** the four stacks are `(6,32,32)` in `ugrizy` order, matching v21. Each predicted PSF is returned and saved even though the color grid only uses predicted object images.
- **Fallback:** all six requested bands are supplied by exact band or nearest **successfully retrieved image+PSF** band. Color comparisons involving substituted bands are not independent g/r/i measurements; RGB fallback is labeled in the grid.
- **Linear RGB:** use fixed, independent `DISPLAY_VMIN` / `DISPLAY_VMAX` for each of the `i`, `r`, `g` bands; the same band limits apply to every candidate and every batch. There is no percentile normalization or asinh stretch. Adjust limits manually to rebalance the displayed colors.
- **PSF/model caveats:** a real model may output different flux scaling or backgrounds; confirm that output channels are in comparable calibrated units before interpreting colors. No per-band photometric recalibration or PSF matching is introduced here.
- **Cache reuse:** `LOAD_NPZ_IF_EXISTS=True` reads saved six-band observed/predicted arrays and fallback mappings when target coordinates match, skipping Rubin retrieval and the model call. `SAVE_NPZ` separately controls new saves. A missing cache still triggers retrieval and prediction. Editing RGB settings in an active notebook kernel requires rerunning only Sections 5–7.
